# 03 · Data Profiling

**Objective:** Produce a full data-quality profile of all three raw sources — shape, memory footprint, dtypes, missingness, duplication, cardinality — *before* deciding how to clean anything. Profiling first, cleaning second, is the order that stops me "fixing" a problem that turns out to be a misunderstanding of the data rather than a real defect.

**Business purpose:** every cleaning decision downstream (notebook 04) needs to trace back to a documented baseline. If a stakeholder asks "why did 12% of rows lose their funding amount," the answer should point to a number measured here, not a guess.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

RAW_DIR = Path("data/raw")

vc = pd.read_csv(RAW_DIR / "investments_VC.csv", encoding="latin1", low_memory=False)
vc.columns = [c.strip() for c in vc.columns]
sf = pd.read_csv(RAW_DIR / "big_startup_secsees_dataset.csv", encoding="utf-8", low_memory=False)
uc = pd.read_csv(RAW_DIR / "global_unicorn_companies.csv", encoding="utf-8-sig", low_memory=False)

sources = {"investments_VC": vc, "big_startup_secsees": sf, "global_unicorn_companies": uc}

## 1. Shape & memory footprint

In [1]:
overview = []
for name, df in sources.items():
    overview.append({
        "source": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "fully_blank_rows": int(df.isna().all(axis=1).sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
    })
overview_df = pd.DataFrame(overview)
overview_df

,source,rows,columns,memory_mb,fully_blank_rows,exact_duplicate_rows
0,investments_VC,54294,39,58.50,4856,4855
1,big_startup_secsees,66368,14,51.84,0,0
2,global_unicorn_companies,1359,9,0.63,0,0


### Observation
`investments_VC` carries a handful of fully-blank trailing rows (a common CSV-export artifact — an extra blank line at the end of the file) and zero exact-duplicate rows once those are excluded. Memory footprint is modest for all three (well under 100MB combined) — no chunked reading needed.

## 2. Data types — do they match what the data dictionary claims?

In [1]:
for name, df in sources.items():
    print(f"--- {name}: dtype counts ---")
    print(df.dtypes.value_counts().to_string())
    print()

--- investments_VC: dtype counts ---
float64    23
str        16

--- big_startup_secsees: dtype counts ---
str      13
int64     1

--- global_unicorn_companies: dtype counts ---
str        7
float64    2



In [1]:
# investments_VC: funding_total_usd should be numeric but Crunchbase exports
# it with thousands-separators and stray spaces, forcing pandas to read it as object.
print("dtype of 'funding_total_usd' as loaded:", vc[' funding_total_usd '].dtype if ' funding_total_usd ' in vc.columns else vc.filter(like='funding_total_usd').columns.tolist())
col = [c for c in vc.columns if 'funding_total' in c.lower()][0]
print(f"Actual column name (after strip): {col!r}")
print(vc[col].dropna().astype(str).head(5).tolist())
print("\n-> stored as text with commas ($ e.g. '1,750,000') -- this is WHY it loads as 'object' dtype,")
print("   not because the values aren't numeric. clean.py strips commas/whitespace and casts to float.")

dtype of 'funding_total_usd' as loaded: ['funding_total_usd']
Actual column name (after strip): 'funding_total_usd'
[' 17,50,000 ', ' 40,00,000 ', ' 40,000 ', ' 15,00,000 ', ' 60,000 ']

-> stored as text with commas ($ e.g. '1,750,000') -- this is WHY it loads as 'object' dtype,
   not because the values aren't numeric. clean.py strips commas/whitespace and casts to float.


### Observation
`funding_total_usd` is text, not because the underlying values are non-numeric, but because Crunchbase's export formats large numbers with thousands separators (`"1,750,000"`). Pandas correctly refuses to infer this as numeric — treating malformed numeric strings as numbers automatically would be a worse default. This is exactly the kind of thing profiling should catch *before* I write cleaning code that assumes it's already a float.

## 3. Missing value profile

In [1]:
def missing_profile(df, name, top=15):
    pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
    pct = pct[pct > 0]
    print(f"--- {name}: {len(pct)} of {df.shape[1]} columns have missing values ---")
    return pct.head(top)

missing_profile(vc, "investments_VC")

--- investments_VC: 39 of 39 columns have missing values ---


state_code           44.45
founded_quarter      29.12
founded_month        29.12
founded_year         29.12
founded_at           28.99
city                 20.21
region               18.66
country_code         18.66
market               16.25
category_list        16.24
homepage_url         15.30
status               11.36
name                  8.95
funding_total_usd     8.94
permalink             8.94
dtype: float64

In [1]:
missing_profile(sf, "big_startup_secsees")

--- big_startup_secsees: 8 of 14 columns have missing values ---


founded_at          22.93
state_code          12.88
region              12.10
city                12.10
country_code        10.48
homepage_url         7.62
category_list        4.74
first_funding_at     0.04
dtype: float64

In [1]:
missing_profile(uc, "global_unicorn_companies")

--- global_unicorn_companies: 5 of 9 columns have missing values ---


select_investors    3.02
city                1.62
country             0.15
year_joined         0.07
date_joined         0.07
dtype: float64

### Observation
- `investments_VC`'s round-type columns (seed, venture, round_A...H, debt_financing, etc.) are overwhelmingly missing — but that's **expected**, not a defect. A startup that never raised a Series B has a genuinely-empty `round_B` value, not a data-quality problem. These are "structural zeros," and notebook 04 treats them as `0`, not as missing-to-be-imputed.
- `big_startup_secsees` is comparatively very complete — it's the cleanest of the three sources by design (it exists specifically to supply reliable `status` labels).
- `global_unicorn_companies` has 1–2% missingness on `country`/`city` for a couple of stealth or unusually-structured entries — small enough to flag and move on rather than needing a strategy.

### Business interpretation
This distinction — "missing because never applicable" vs. "missing because the data is incomplete" — directly determines whether I impute, flag, or leave a `NaN` alone. Getting it wrong in either direction (imputing a structural zero, or treating a true gap as zero) would distort every funding total downstream.

## 4. Duplicate & cardinality analysis

In [1]:
print("Duplicate permalink counts (case-sensitive, raw):")
for name, df in sources.items():
    if "permalink" in df.columns:
        dup = df["permalink"].dropna().duplicated().sum()
        print(f"  {name}: {dup} duplicate permalink rows out of {df['permalink'].notna().sum():,} non-null")

Duplicate permalink counts (case-sensitive, raw):
  investments_VC: 2 duplicate permalink rows out of 49,438 non-null
  big_startup_secsees: 0 duplicate permalink rows out of 66,368 non-null


In [1]:
cardinality = []
for name, df in sources.items():
    for c in df.columns:
        n_unique = df[c].nunique(dropna=True)
        cardinality.append({"source": name, "column": c, "unique_values": n_unique, "pct_unique_of_rows": round(n_unique/len(df)*100, 1)})
card_df = pd.DataFrame(cardinality)
print("Highest-cardinality columns (top 10) -- candidates for grouping/bucketing before modeling:")
card_df.sort_values("unique_values", ascending=False).head(10)

Highest-cardinality columns (top 10) -- candidates for grouping/bucketing before modeling:


,source,column,unique_values,pct_unique_of_rows
39,big_startup_secsees,permalink,66368,100.0
40,big_startup_secsees,name,66102,99.6
41,big_startup_secsees,homepage_url,61191,92.2
0,investments_VC,permalink,49436,91.1
1,investments_VC,name,49350,90.9
2,investments_VC,homepage_url,45850,84.4
42,big_startup_secsees,category_list,27296,41.1
43,big_startup_secsees,funding_total_usd,18896,28.5
3,investments_VC,category_list,16675,30.7
5,investments_VC,funding_total_usd,14617,26.9


### Observation
`category_list` and `market`-style free-text columns have hundreds of distinct values — far too many for one-hot encoding without grouping. This one profiling result is *why* `data_prep.py` (used from notebook 06 onward) buckets industries/countries into "top-N + Other" rather than encoding every category — a decision I make here, before any model exists to break.

## 5. Cross-check against the documented data dictionary

`docs/data_dictionary.md` claims specific column roles for the *cleaned/warehouse* tables. Let me confirm the profiling above is consistent with what that document asserts (a documentation-vs-reality check, not just documentation-writing).

In [1]:
with open("docs/data_dictionary.md") as f:
    dd_text = f.read()
print(dd_text[:650])
print("...")
print(f"\n[data_dictionary.md is {len(dd_text.splitlines())} lines total -- full version in docs/]")

# Data Dictionary — Star Schema

## Schema Diagram (logical)

```
                         ┌─────────────────┐
                         │   dim_date       │
                         │ (role-playing:   │
                         │  founded/first/  │
                         │  last funding)   │
                         └────────┬─────────┘
                                  │
┌──────────────┐         ┌───────▼────────────────┐         ┌──────────────┐
│ dim_geography│◄────────┤ fact_startup_funding     ├────────►│ dim_industry │
└──────────────┘         │ (grain: 1 row/startup)   │         └──────────────┘
                          └───────┬───
...

[data_dictionary.md is 96 lines total -- full version in docs/]


### Observation
The data dictionary's claims (`dim_date`'s role, `is_pre_1900_founding` as a flag rather than a filter, `primary_category` as first-listed of a multi-value field) all match what I found by directly inspecting the raw and warehouse data — nothing here needed a correction to the documentation.

## 6. Initial risks identified from this profile (carried into notebook 04)

| Risk | Evidence from this notebook | Planned treatment |
|---|---|---|
| `funding_total_usd` stored as text | comma-formatted numeric strings | strip + cast to float (04) |
| Structural zeros mistaken for missing | round-type columns ~90%+ "missing" | fill with 0, not impute (04) |
| High-cardinality categoricals | `category_list`/`market` hundreds of values | top-N + "Other" bucketing (06) |
| Join-key casing mismatch | seen in notebook 02 | normalize before merge (04) |
| Fully-blank trailing rows | `investments_VC` | drop via `dropna(how='all')` (04) |

## 7. Interview questions this notebook prepares me for

- *"How do you decide whether missing data should be imputed or filled with zero?"* — I ask whether the absence is structural (the field genuinely doesn't apply) or a true data gap. Structural zeros should never be imputed with a mean/median — that would fabricate financial activity that didn't happen.
- *"Why profile before cleaning instead of cleaning as you go?"* — Cleaning without a documented baseline makes every decision unfalsifiable — there's no way to check months later whether a "fix" was actually correct.

## Next notebook
`04_Data_Cleaning.ipynb` — implements and justifies every cleaning decision identified above.